# Dataframer: Robert Fagles's Odyssey to Pandas DF

### [—————————————pipeline—————————————]
### »——raw—»—clean—»—normalize—»—DATAFRAME——»

Here are some transformation and frequencies for future exploratory analysis of Green's Odyssey.

Columns: author, year, title, book_num, text, num_lines, num_sentences, num_words, 


In [1]:
# library imports
import os

import numpy as np
import pandas as pd

import re
import nltk

In [2]:
# Display options
pd.set_option("display.max_colwidth", None)  # Prevent truncation of long values
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.expand_frame_repr", False)  # Prevent wrapping in DataFrames

In [3]:
# Visualization libraries
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/Users/debr/English-Homer') 
import bard_visualization as viz# This will apply the visualization settings
from bard_visualization import color_palette 
sys.path.append('/Users/debr/English-Homer/functions') 
import e_nlp as e

Functions for NLP are live! use e.<function> to call them.
Download complete.
Stopwords customized:
  Added: {'seven', 'one', 'n', 'nine', "'", 'eight', 'ten', 'three', 'five', 'four', "'and", 'six', 'two'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'“', '—', '…', '\\', '”', '-', '’', '‘'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
# TO UPDATE
translator = "Fagles"
book_breaker = "Book"

# Check Paths
filepath = f"/Users/debr/odysseys_en/Normalized_txts/Odyssey_{translator}_Normalized.txt"
output_path = f"/Users/debr/English-Homer/dataframers_by_author/{translator}_DFed/"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_path_plots = f"{output_path}/plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)

# READING FILE TO extracted_lines
with open(filepath, 'r') as file:
    extracted_lines = file.readlines()

text = "".join(extracted_lines)

In [5]:
# Function to split the text into books

#books = string_into_books(text, book_breaker)
books = e.list_into_books(extracted_lines, book_breaker)

# Verify the results
print(f"Found {len(books)} books")
for i, book in enumerate(books):
    print(f"Book {i+1} starts with: {book[0]}")
    print(f"Book {i+1} has {len(book)} lines")

Found 24 books
Book 1 starts with: Sing to me of the man, Muse, the man of twists and turns …

Book 1 has 534 lines
Book 2 starts with: When young Dawn with her rose-red fingers shone once more

Book 2 has 493 lines
Book 3 starts with: As the sun sprang up, leaving the brilliant waters in its wake,

Book 3 has 573 lines
Book 4 starts with: At last they gained the ravines of Lacedaemon ringed by hills

Book 4 has 978 lines
Book 5 starts with: As Dawn rose up from bed by her lordly mate Tithonus,

Book 5 has 558 lines
Book 6 starts with: So there he lay at rest, the storm-tossed great Odysseus,

Book 6 has 376 lines
Book 7 starts with: Now as Odysseus, long an exile, prayed in Athena’s grove,

Book 7 has 406 lines
Book 8 starts with: When young Dawn with her rose-red fingers shone once more

Book 8 has 677 lines
Book 9 starts with: Odysseus, the great teller of tales, launched out on his story:

Book 9 has 649 lines
Book 10 starts with: “We reached the Aeolian island next, the home of Ae

In [6]:
# Books (lists) into DataFrame
df = e.book_into_df(f"{translator}", "1996", "The Odyssey", books)

# Apply functions & add new columns
df['num_lines'] = df['text'].apply(e.count_lines)
df['num_sentences'] = df['text'].apply(e.count_sentences)
df['num_words'] = df['text'].apply(e.count_words)

In [7]:
# Initialize the pipeline
nlp = e.NLPPipeline(language='english')

# Customize stopwords
nlp.customize_stopwords(
    include={'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
             "'", 'n', "'and",},
    exclude={''}
)
# Customize punctuation
nlp.customize_punctuation(
    keep={'-', ""},  # Keep hyphens and apostrophes
    remove={r'…', '—','”','’','“', '‘', '-', '\\'}  # Additional characters to remove
)

# Process with default pipeline (lowercase -> tokenize -> remove punctuation -> remove stopwords)
df = nlp.process_dataframe(df, 'text', 'tokens')
df['num_tokens'] = df['tokens'].map(len)

Stopwords customized:
  Added: {'seven', 'one', 'n', 'nine', "'", 'eight', 'ten', 'three', 'five', 'four', "'and", 'six', 'two'}
  Removed: {''}
  Total stopwords: 211
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'“', '—', '…', '\\', '”', '-', '’', '‘'}
  Punctuation to be removed: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~—‘’“”…


In [8]:
print('Type:', type(df['text'][0]))
print('Lenght:',len(df['text'][0]))
print(df['text'][:6])

Type: <class 'list'>
Lenght: 534
0                                                                                   [Sing to me of the man, Muse, the man of twists and turns …\n, driven time and again off course, once he had plundered\n, the hallowed heights of Troy.\n, Many cities of men he saw and learned their minds,\n, many pains he suffered, heartsick on the open sea,\n, fighting to save his life and bring his comrades home.\n, But he could not save them from disaster, hard as he strove—\n, the recklessness of their own ways destroyed them all,\n, the blind fools, they devoured the cattle of the Sun\n, and the Sungod blotted out the day of their return.\n, Launch out on his story, Muse, daughter of Zeus,\n, start from where you will—sing for our time too.\n, By now,\n, all the survivors, all who avoided headlong death\n, were safe at home, escaped the wars and waves.\n, But one man alone …\n, his heart set on his wife and his return—Calypso,\n, the bewitching nymph, the lustrous 

In [9]:
print('Type:', type(df['tokens'][0]))
print('Lenght:',len(df['tokens'][0]))
print(df['tokens'][:6])

Type: <class 'list'>
Lenght: 2326
0                                                                                 [sing, man, muse, man, twists, turns, driven, time, course, plundered, hallowed, heights, troy, many, cities, men, saw, learned, minds, many, pains, suffered, heartsick, open, sea, fighting, save, life, bring, comrades, home, could, save, disaster, hard, strove, recklessness, ways, destroyed, blind, fools, devoured, cattle, sun, sungod, blotted, day, return, launch, story, muse, daughter, zeus, start, sing, time, survivors, avoided, headlong, death, safe, home, escaped, wars, waves, man, alone, heart, set, wife, return, calypso, bewitching, nymph, lustrous, goddess, held, back, deep, arching, caverns, craving, husband, wheeling, seasons, brought, year, around, year, spun, gods, reach, home, ithaca, though, even, would, free, trials, even, ...]
1                                        [young, dawn, rosered, fingers, shone, true, son, odysseus, sprang, bed, dressed, shoulde

In [10]:
#Boolean check for missing values
e.check_df(df)

No missing values

df columns: Index(['author', 'year', 'title', 'book_num', 'text', 'num_lines',
       'num_sentences', 'num_words', 'tokens', 'num_tokens'],
      dtype='object') 

Shape: (24, 10)


In [12]:
df

author  year        title  book_num                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [11]:
# Create output directory if it doesn't exist
output_filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{translator}_DataFrame.csv"
os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

# save df to csv
df.to_csv(output_filepath, index=False)

print(f"Normalization complete. File saved to: {output_filepath}")

Normalization complete. File saved to: /Users/debr/odysseys_en/dataframed/Odyssey_Fagles_DataFrame.csv
